2/23/26  

I want to create a Missouri map of streamflow percentiles. I am using [this](https://doi-usgs.github.io/hyswap/examples/site_conditions_examples.html) tutorial. Unfortunately, it looks like the tutorial is using the old method for `dataretrieval`.  

I plan to use the old method and see if I can get it up and running. Then, I will come back and update to the current API method. 


#### TODO: Undo overwrite of "today" variable
#### TODO: Finish updating the local data file update code to check last date of file contents instead of file name. 
#### TODO: Add ability to plot gages without 30 yrs of data?
#### TODO: Update to new NWIS method
#### TODO: Move my USGS Token to external file and add to .gitignore 

07189000	

---
### Percentile Definitions
`percentile_level` = percentile levels from 0-100 at which to calculate values  
`percentile_threshold` = calculated values at each percentile level using approved data  
`percentile` = interpolated between percentile_threshold to estimate percentile of newly recorded and provisional data

In [2]:
import os

with open(r'..\usgs_api_key.txt', 'r') as f:
    api_key = f.readline()
    os.environ["API_USGS_PAT"] = api_key

In [2]:
import os
import glob
from dataretrieval import nwis
import hyswap
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# from tqdm import tqdm # used for progress bar indicators
import geopandas # has dependencies of folium and mapclassify to create maps within this notebook
# import warnings
import folium

In [3]:
today = str(datetime.today().date())
today = '2026-03-02'

## Define Helper Functions

In [4]:
# Data QAQC function for provisional NWIS data
def qaqc_usgs_data(df, data_column_name):
    #replace invalid -999999 values with NA
    try:
        df[data_column_name] = df[data_column_name].replace(-999999, np.nan)
    except KeyError:
        print(f"Does not have column {data_column_name}")
    # add any additional QAQC steps needed
    return df

In [5]:
## Copied from tutorial:

def create_gage_condition_map(gage_df, flow_data_col, map_schema, streamflow_data_type) -> folium.Map:
        # Format date and set to str type for use in map tooltips
        if flow_data_col == '00060':
                gage_df['Date'] = gage_df['datetime'].dt.strftime('%Y-%m-%d %H:%M')
        elif flow_data_col == '00060_Mean':
                gage_df['Date'] = gage_df['datetime'].dt.strftime('%Y-%m-%d')
        gage_df = gage_df.drop('datetime', axis=1)
        # create colormap for map from hyswap schema
        schema = hyswap.utils.retrieve_schema(map_schema)
        flow_cond_cmap = schema['colors']
        if 'low_color' in schema:
                flow_cond_cmap = [schema['low_color']] + flow_cond_cmap
        if 'high_color' in schema:
                flow_cond_cmap = flow_cond_cmap + [schema['high_color']]
        # if creating a drought map, set handling of non-drought flows
        if map_schema in ['WaterWatch_Drought', 'NIDIS_Drought']:
                gage_df['flow_cat'] = gage_df['flow_cat'].cat.add_categories('Other')
                gage_df.loc[gage_df['flow_cat'].isnull(), 'flow_cat'] = 'Other'
                flow_cond_cmap = flow_cond_cmap + ['#e3e0ca'] # light taupe
        # set NA values to "Not Ranked" category
        gage_df['flow_cat'] = gage_df['flow_cat'].cat.add_categories('Not Ranked')
        gage_df.loc[gage_df['est_pct'].isna(), 'flow_cat'] = 'Not Ranked'
        flow_cond_cmap = flow_cond_cmap + ['#d3d3d3'] # light grey
        # renaming columns with user friendly names for map
        gage_df = gage_df.rename(columns={flow_data_col:'Discharge (cfs)',
                                                'est_pct':'Estimated Percentile',
                                                'site_no':'USGS Gage ID',
                                                'station_nm':'Streamgage Name',
                                                'flow_cat':'Streamflow Category'})
        # convert dataframe to geopandas GeoDataFrame
        gage_df = geopandas.GeoDataFrame(gage_df,
                             geometry=geopandas.points_from_xy(gage_df.dec_long_va,
                                                               gage_df.dec_lat_va),
                             crs="EPSG:4326").to_crs("EPSG:5070")
        # Create map        
        m = folium.Map(
                    location=(38.36768, -92.47729), 
                    zoom_start=7, 
                    # tiles="Cartodb Positron",
                    # tiles = "OpenStreetmap",
                    # tiles = "Stadia.Outdoors",
                    # tiles = "OpenTopoMap",
                    # tiles="CartoDB Voyager",
                    # tiles=None,
        )
        m = gage_df.explore(
                            column="Streamflow Category",
                                cmap=flow_cond_cmap,
                                tooltip=["USGS Gage ID", "Streamgage Name", "Streamflow Category", "Discharge (cfs)", "Estimated Percentile", "Date", "record_length_yr"],
                                tiles="CartoDB Positron",
                                marker_kwds=dict(radius=5),
                                legend_kwds=dict(caption=streamflow_data_type + '<br> Streamflow  Category'))
        return m #returns a folium map object

## Find Sites

Update to modern NWIS API  
https://github.com/DOI-USGS/dataretrieval-python/blob/main/demos/WaterData_demo.ipynb

**UPDATE: The tutorial is now updated with the waterdata API. Can use it to see how they recommend making the switch. 

In [5]:
def get_usgs_gage_metadata():
    
    state = 'MO'
    # Query NWIS for what streamgage sites were active within the last week
    sites, _ = nwis.what_sites(
        stateCd=state, 
        parameterCd=['00060'], 
        # period="P1W"
        startDt=f'{today[:4]}-01-01'
    ) #, siteType='ST')
    
    # display(sites)
    return sites

sites = get_usgs_gage_metadata()

## Load Time Series Data

In [9]:
# create a python dictionary of dataframes by site id number
# Want to save downloaded data so don't have to do this download every time. 
# If data is saved locally, load it from there. 

def get_usgs_daily_api(site_no, start='1850-01-01', end=today, pcode='00060'):
        
    df = nwis.get_record(
            sites=site_no, 
            parameterCd=pcode, 
            start=start, 
            end=today, 
            service='dv'
        ).drop('site_no', axis=1) # don't need to save the extra data if it's in the file name. Can add back later if needed. 

    return df

def load_local_data(fn):
    return pd.read_csv(fn, index_col='datetime', parse_dates=['datetime'])

def update_local_data(fn_local, fn_today, today):
    '''
    Update local data if gage data already exists and adding recent data to it. 
        Loads local data as df
        Downloads only recent data from USGS API
        Combine dfs
        Delete old fn
        Write data to new fn
        Return df 
    '''
    
    df_local = load_local_data(fn_local)
    
    # last_local_date = fn_site.split('_')[-1].strip('.csv')
    last_local_date = df_local.index[-1]

    if last_local_date < today:
        df_new = get_usgs_daily_api(site_no, start=last_local_date)     ## add end = yesterday to get rid of potential date mixup       

        df = pd.concat([df_local, df_new])
        os.remove(fn_site)
        df.to_csv(fn_today)

    else: 
        df = df_local
        
    return df


## Break this into tow differnt functions. One that udpates local data. Another that loads data. 

def get_flow_data_time_series(sites: pd.DataFrame) -> dict[str, pd.DataFrame]:
    '''
    sites: df is output of nwis.what_sites()

    output: dictionary keys is string site number with df time series of all daily data for the site

    first run will download all data from USGS API and save locally. 
    subsequent runs will check for local data and  request only new data from USGS API if needed. 
    '''

    flow_data = {}
    
    path_data = r'..\data\daily'
    
    for site_no in sites['site_no'].iloc[-3:]:
    
        fn_today = os.path.join(path_data, site_no + f'_{today}.csv')
        glob_site = glob.glob(os.path.join(path_data, site_no + '*csv'))

        print(fn_today)
    
        ## Check if gage & today's date exist
        if os.path.isfile(fn_today):
            print('1')
            df = load_local_data(fn_today)
            # flow_data[site_no] = df
            # continue
            
        ## Check if gage but not today's date exist. If so, only update data. Don't redownload the whole thing. 
        elif len(glob_site) > 0:
            print('2')
            fn_local = glob_site[-1]
            df = update_local_data(fn_local)
            
        else:
            print('3')
            df = get_usgs_daily_api(site_no)
            df.to_csv(fn_today)

        # if not df.empty:        
        # flow_data[site_no] = qaqc_usgs_data(df, '00060_Mean') # breaks with an empty. WHen do I need to do qaqc?
        flow_data[site_no] = df

        
    return flow_data

flow_data = get_flow_data_time_series(sites)


#%% Hack to load data without making a call to nwis.what_sites to get the gages

def get_sites_local():
    path_data = r'..\data\daily'
    sites_lst = []
    for fn in os.listdir(path_data):
        if '.csv' in fn:
            sites_lst.append(fn.split('_')[0])
    return sites_lst

# sites_lst = get_sites_local()
# flow_data = get_flow_data_time_series(pd.DataFrame(sites_lst, columns=['site_no']))

    

..\data\daily\07188885_2026-03-02.csv
1
..\data\daily\07189000_2026-03-02.csv
1
..\data\daily\07189100_2026-03-02.csv
1


## Percentiles

In [52]:
# Define what percentile levels (thresholds) that we want to calculate.
# Intervals of 5 or less are recommended to have sufficient levels to interpolate between in later calculations.
# Note that 0 and 100 percentile levels are ignored, refer to min and max values returned instead.
percentile_levels = np.concatenate((np.array([1]), np.arange(5,96,5), np.array([99])), axis=0)
print(percentile_levels)

[ 1  5 10 15 20 25 30 35 40 45 50 55 60 65 70 75 80 85 90 95 99]


## N-Day Daily Percentile Calculations
Edited USGS code to only calculate percentiles for the day of interest. Runs much faster. Don't need percentile info for every day of year for this appication.

In [77]:
def get_recent_values(flow_data, today, day=1) -> pd.DataFrame:
    '''
    Get last n days worth of data to use as "current". Get rolling average

    today = str of day data were collected. So typically last day of data should be from "yesterday"
    
    day = 1, 7, 14, or 24

    returns df with single, {day}-day averaged value for each gage. 
    '''

    assert day in [1, 7, 14, 28], 'Only for 1, 7, 14, or 28 day averages'    

    
    stop = yesterday = datetime.strptime(today, '%Y-%m-%d') - timedelta(1)
    start = yesterday
    if day > 1:
        start = yesterday - timedelta(day-1) #-1 bc both ends are inclusive in date index

    recent_dvs = pd.DataFrame()
    for site_no, df in flow_data.items():

        df = qaqc_usgs_data(df, '00060_Mean')
            
        # df = qaq
        if not df.empty:  
            df['site_no'] = site_no
            df_rolled = hyswap.utils.rolling_average(
                # df.iloc[-day:],   # initial method
                df.loc[str(start):str(yesterday)],  #use dates explicitly in case last date not consistent (this has happened) 
                '00060_Mean', 
                f'{day}D').dropna()  ## change from "7D". Make sure this works as expected. 
            recent_dvs = pd.concat([recent_dvs, df_rolled], axis=0)
    
    recent_dvs.set_index('site_no', append=True, inplace=True)
    recent_dvs.swaplevel('site_no', 'datetime')
    
    # recent_dvs = qaqc_usgs_data(recent_dvs, '00060_Mean')


    ## THis was originally to put into nwis.get_record. It assumes new data is really collected "today". Not alwasys true. "today" in code can be previous day. 
    # yesterday = datetime.strftime(datetime.now(tz=ZoneInfo("US/Eastern")) - timedelta(1), '%Y-%m-%d')
    # start = yesterday
    # if day > 1:
    #     start = datetime.strftime(datetime.now(tz=ZoneInfo("US/Eastern")) - timedelta(day), '%Y-%m-%d')  
        
    # past_dvs = nwis.get_record(
    #     sites=sites['site_no'].tolist(),
    #     parameterCd='00060',
    #     start=start,
    #     end=yesterday,
    #     service='dv',
    #     multi_index=False
    # )

    # recent_dvs = qaqc_usgs_data(recent_dvs, '00060_Mean')
    
    return recent_dvs

In [78]:
day = 28
past_dvs_nd = get_recent_values(flow_data, today, day)
past_dvs = get_recent_values(flow_data, today, 1)


## Make sure to figure out the issues with teh data & function above before doing this long term.
## This causes a problem once you get to percentile interpolation step
# past_dvs = past_dvs.droplevel(level='datetime')
# past_dvs_7d = past_dvs_7d.droplevel(level='datetime')



Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean
Does not have column 00060_Mean


In [ ]:
### testing

day = 7
site_no = '07189100'

stop = yesterday = datetime.strptime(today, '%Y-%m-%d') - timedelta(1)
start = yesterday
if day > 1:
    start = yesterday - timedelta(day-1)

display(flow_data[site_no].loc[str(start):str(yesterday)])

hyswap.utils.rolling_average(
                # flow_data[site_no].iloc[-day:], 
                flow_data[site_no].loc[str(start):str(yesterday)],
                '00060_Mean', 
                f'{day}D') #.dropna() 

In [ ]:
## Get rolling n day average for all data at each site. To be used for percentile calcs
## THIS IS CURRENTLY UNUSED

flow_data_nday = {}
for site_no, df in flow_data.items():
    if not df.empty:
        flow_data_nday[site_no] = hyswap.utils.rolling_average(df, '00060_Mean', f'{day}D').round(2)

    

In [55]:

## DELTE AFTER CONFIRMING THIS ISNT" NEEDED

# def calculate_single_day_percentile(df, value, today):
#     '''
#     input is [df time series of all data for single gage, value want percentile for, date as str for last data update]

#     output is percentile
#     '''
    
#     # yesterday_md = str(datetime.today().date() - timedelta(days=1))[-5:]
#     yesterday_md = str(datetime.strptime(today, '%Y-%m-%d').date() - timedelta(1))[-5:]
    
#     s_doy = hyswap.filter_data_by_month_day(df, yesterday_md, '00060_Mean')
#     # display(s_doy)
    
#     ptile_thresholds = hyswap.calculate_fixed_percentile_thresholds(s_doy, percentiles=percentile_levels)
#     # display(ptile_thresholds)            
    
#     ptile = hyswap.calculate_fixed_percentile_from_value(value, ptile_thresholds)
#     print(f"{value}: {ptile}%, n={ptile_thresholds["count"].values}")

#     return ptile
    

## THIS IS THE FUNCTION THAT GREATLY REDUCES THE TIME IN THE SLOW PART OF THE CODE
def calculate_single_day_percentile_thresholds(df, today):
    '''
    input is [df time series of all data for single gage, value want percentile for, date as str for last data update]

    output is df of thresholds
    '''
    
    # yesterday_md = str(datetime.today().date() - timedelta(days=1))[-5:]
    yesterday_md = str(datetime.strptime(today, '%Y-%m-%d').date() - timedelta(1))[-5:]
    
    s_doy = hyswap.filter_data_by_month_day(df, yesterday_md, '00060_Mean')
    # display(s_doy)
    
    ptile_thresholds = hyswap.calculate_fixed_percentile_thresholds(s_doy, percentiles=percentile_levels)
    # ptile_thresholds.index = list(yesterday_md)
    # display(ptile_thresholds)            
    
    # ptile = hyswap.calculate_fixed_percentile_from_value(value, ptile_thresholds)
    # print(f"{value}: {ptile}%, n={ptile_thresholds["count"].values}")

    return ptile_thresholds


In [ ]:
sites_lst

In [56]:
## calculate percentiles of n day rolling avreaged time series data   

def get_percentiles(sites, flow_data, df_dvs, today) -> dict[str, pd.DataFrame]:
    '''
    flow_data = dict 
    df_dvs = output of 'get_recent_values'

    output = Dict
          key: site_no
          value: pd.DataFrame of percentiles for single day
    '''

    percentile_thresholds = {}
    for site_no in sites['site_no']:
 #    for site_no in ['06818000', '07185910',
 # '07186000',
 # '07187000',
 # '07188653',
 # '07188838',
 # '07188885',
 # '07189000',
 # '07189100']:
    
        if flow_data[site_no].empty:
            print(f'{site_no} does not have daily data')
            continue
            
        if (flow_data[site_no].index[-1] - flow_data[site_no].index[0]) / timedelta(days=365) < 30: 
            print(f'{site_no} does not have 30 years of data')
            continue
        
        if '00060_Mean' in flow_data[site_no].columns:
            print(f'{site_no} precentile calculated')
            # Filter data as only approved data in NWIS should be used to calculate statistics
            df = hyswap.utils.filter_approved_data(flow_data[site_no], '00060_Mean_cd')

            ## Original method calculating percentile thresholds for each day of year
            # percentile_thresholds[site_no] = hyswap.percentiles.calculate_variable_percentile_thresholds_by_day(
            #     df, '00060_Mean', percentiles=percentile_levels)

            ## New method to only calculate for the current day of year of interest
            percentile_thresholds[site_no] = calculate_single_day_percentile_thresholds(df, today)        
            
        else:
            print('No standard discharge data column found for site ' + site_no + ', skipping')

    return percentile_thresholds
    


def get_years_used_for_percentile_calcs(percentile_values) -> dict[str, int]:
    percentile_year_count = {}
    for site_no in percentile_values.keys():
        percentile_year_count[site_no] = percentile_values[site_no]['count'].max()
    return percentile_year_count

# percentile_values = get_percentiles(sites, flow_data)
# percentile_year_count = get_years_used_for_percentile_calcs(sites, percentile_values)



---
Running again with 7 day data. Same speed issues?  
Yes very similar results with 7 day data. 

`CPU times: total: 3min 2s`  
`Wall time: 3min 46s`

In [79]:
%%time
percentile_values = get_percentiles(sites, flow_data, past_dvs, today)
# percentile_values = get_percentiles(pd.DataFrame(sites_lst, columns=['site_no']), flow_data, past_dvs, today)


05490600 precentile calculated
05495000 precentile calculated
05496000 precentile calculated
05497150 does not have 30 years of data
05498150 does not have 30 years of data
05498700 does not have 30 years of data
05500000 precentile calculated
05501000 precentile calculated
05502000 precentile calculated
05502300 precentile calculated
05502500 precentile calculated
05503800 precentile calculated
05504800 precentile calculated
05506100 precentile calculated
05506350 does not have 30 years of data
05506800 precentile calculated
05507600 precentile calculated
05507800 precentile calculated
05508000 precentile calculated
05508805 precentile calculated
05514500 precentile calculated
05514840 does not have 30 years of data
05514860 does not have 30 years of data
06813000 precentile calculated
06817500 precentile calculated
06817700 precentile calculated
06818000 precentile calculated
06819500 precentile calculated
06819550 does not have 30 years of data
06820410 does not have 30 years of dat

In [80]:
percentile_year_count = get_years_used_for_percentile_calcs(percentile_values)

In [84]:
## Use percentile thresholds to interpolate the percentile value for the new value. 
## Edit variable names so that this makes more sense. 

# def estimate_percentile_of_recent_value():

df = pd.DataFrame()
for site_no, site_df in past_dvs.groupby(level="site_no", group_keys=True):  
    print(f'{site_no} {len(site_df)}')
    if site_no not in list(percentile_values.keys()):
        print(f'{site_no} not in percentile_values')
        ## TODO: need to add this somehow into the dataframe so I can plot the gages with less than 30 yrs of data?
        continue
    elif percentile_values[site_no].isnull().all().all():
        print(f'{site_no} all values Null')
        continue
    # percentiles = hyswap.percentiles.calculate_multiple_variable_percentiles_from_values(
    #     site_df,'00060_Mean', percentile_values[site_no])

    
    if site_df['00060_Mean'].isna().values[-1]:
        print(f'{site_no} past_dvs_nd is Nan')
        continue 
        
    percentile = hyswap.percentiles.calculate_fixed_percentile_from_value(
        site_df['00060_Mean'], percentile_values[site_no])

    site_df['est_pct'] = percentile

    # if site_no == '05502300':
    #     break

    # percentiles = pd.DataFrame({'value': site_df['00060_Mean']
    
        
    df = pd.concat([df, site_df])

05498700 1
05498700 not in percentile_values
05501000 1
05502000 1
05502300 1
05502500 1
05503800 1
05504800 1
05506100 1
05506350 1
05506350 not in percentile_values
05506800 1
05507600 1
05507800 1
05508000 1
05508805 1
05514500 1
05514840 1
05514840 not in percentile_values
05514860 1
05514860 not in percentile_values
06817500 1
06818000 1
06820500 1
06821080 1
06821080 not in percentile_values
06821150 1
06821190 1
06821250 1
06821250 not in percentile_values
06893000 1
06893150 1
06893150 not in percentile_values
06893500 1
06893557 1
06893557 not in percentile_values
06893578 1
06893578 not in percentile_values
06893820 1
06893820 not in percentile_values
06893830 1
06893830 not in percentile_values
06893940 1
06893940 not in percentile_values
06893970 1
06893970 not in percentile_values
06894000 1
06894200 1
06894200 not in percentile_values
06894650 1
06894650 not in percentile_values
06895000 1
06895500 1
06896000 1
06896400 1
06896400 not in percentile_values
06896900 1
06896

In [ ]:
df

In [85]:
# categorize streamflow by the estimated streamflow percentiles
df = hyswap.utils.categorize_flows(df, 'est_pct', schema_name="NWD")
df = df.reset_index(level='datetime')
# Prep Data for mapping by joining site information and flow data`
gage_df = pd.merge(sites, df, how="right", on="site_no")

# Add year count to gage_df
gage_df.set_index('site_no', inplace=True)
gage_df['record_length_yr'] = pd.Series(percentile_year_count)
gage_df.reset_index(inplace=True)

In [ ]:
df

## Plot Maps

In [86]:
m = create_gage_condition_map(gage_df, '00060_Mean', 'NWD', 'Current Daily Mean')
m.fit_bounds([[-95.77, 35.99], [-89.10, 41.61]])
display(m)

# m.save(f'{today}_mo-daily-streamflow-percentiles.html')


In [ ]:
m = create_gage_condition_map(gage_df, '00060_Mean', 'WaterWatch_BrownBlue', 'Current Daily Mean')
display(m)

## To Delete. Get Most Recent Daily Values

In [ ]:
yesterday = datetime.strftime(datetime.now(tz=ZoneInfo("US/Eastern")) - timedelta(1), '%Y-%m-%d')
# recent_dvs = nwis.get_record(sites=list(flow_data.keys()), parameterCd='00060', start=yesterday, end=yesterday, service='dv')
# recent_dvs = qaqc_usgs_data(recent_dvs, '00060_Mean')



In [ ]:
## Updated. The above call was very fast so does this really matter?
recent_dvs = pd.DataFrame()
for site_no, df in flow_data.items():
    if not df.empty:  
        df['site_no'] = site_no
        recent_dvs = pd.concat([recent_dvs, df.iloc[[-1]]], axis=0)

recent_dvs.set_index('site_no', append=True, inplace=True)
recent_dvs = qaqc_usgs_data(recent_dvs, '00060_Mean')


#### Categorize streamflow based on calculated percentile values  
Calculate estimated streamflow percentile for the new data by interpolating against the previously calculated percentile threshold levels.

In [ ]:
%%time
# estimate percentiles


df = pd.DataFrame()
# I think this "groupby" is just procedural as a future function expects a df input. I don't see a purpose for group_keys either.
for site_no, site_df in recent_dvs.groupby(level="site_no", group_keys=False):   
    if site_no not in list(percentile_values.keys()):
        print(f'{site_no} not in perecentile_values')
        ## TODO: need to add this somehow into the dataframe so I can plot the gages with less than 30 yrs of data?
        continue
    elif percentile_values[site_no].isnull().all().all():
        print(f'{site_no} all values Null')
        continue
    percentiles = hyswap.percentiles.calculate_multiple_variable_percentiles_from_values(
        site_df,'00060_Mean', percentile_values[site_no])
    df = pd.concat([df, percentiles])
            


In [ ]:
# categorize streamflow by the estimated streamflow percentiles
df = hyswap.utils.categorize_flows(df, 'est_pct', schema_name="NWD")
df = df.reset_index(level='datetime')
# Prep Data for mapping by joining site information and flow data
gage_df = pd.merge(sites, df, how="right", on="site_no")

# Add year count to gage_df
gage_df.set_index('site_no', inplace=True)
gage_df['record_length_yr'] = pd.Series(percentile_year_count)
gage_df.reset_index(inplace=True)

### TO DELETE

In [ ]:
%%time 
## Calculate specified percentile values for each time series in flow_data dict using only approved data

percentile_values = {}
percentile_year_count = {}
# for site_no in sites['site_no']:
for site_no in flow_data.keys():

    if flow_data[site_no].empty:
        print(f'{site_no} does not have daily data')
        continue
        
    if (flow_data[site_no].index[-1] - flow_data[site_no].index[0]) / timedelta(days=365) < 30: 
        print(f'{site_no} does not have 30 years of data')
        continue
        
    if '00060_Mean' in flow_data[site_no].columns:
        # Filter data as only approved data in NWIS should be used to calculate statistics
        df = hyswap.utils.filter_approved_data(flow_data[site_no], '00060_Mean_cd')
        percentile_values[site_no] = hyswap.percentiles.calculate_variable_percentile_thresholds_by_day(
            df, '00060_Mean', percentiles=percentile_levels)
        percentile_year_count[site_no] = percentile_values[site_no]['count'].max() #iloc[0]
    else:
        print('No standard discharge data column found for site ' + site_no + ', skipping')